In [1]:
import re
import sqlite3
import logging
import pandas as pd
from pathlib import Path

In [2]:
DB_PATH          = "railway.db"
BLOCK_TRIGGER    = "EXCEPTION REPORT"
HEADER_SCAN_ROWS = 12
LOG_FORMAT       = "%(asctime)s [%(levelname)s] %(message)s"
 
logging.basicConfig(level=logging.INFO, format=LOG_FORMAT)
log = logging.getLogger(__name__)

In [3]:
DATASET_TYPE_MAP = [
    ("sod exception",           "sod_data"),
    ("sod_exception",           "sod_data"),
    ("lip flow",                "lip_flow_data"),
    ("lip_flow",                "lip_flow_data"),
    ("vertical wear",           "vertical_wear_data"),
    ("vertical_wear",           "vertical_wear_data"),
    ("lateral rail wear",       "lateral_wear_data"),
    ("lateral wear",            "lateral_wear_data"),
    ("lateral_wear",            "lateral_wear_data"),
    ("sleeper defect",          "sleeper_defects_data"),
    ("sleeper_defect",          "sleeper_defects_data"),
    ("sleeper",                 "sleeper_defects_data"),
    ("rail defect",             "rail_defects_data"),
    ("rail_defect",             "rail_defects_data"),
    ("fittings",                "fittings_data"),
    ("fitting",                 "fittings_data"),
    ("ballast and vegetation",  "ballast_vegetation_data"),
    ("ballast & vegetation",    "ballast_vegetation_data"),
    ("ballast_and_vegetation",  "ballast_vegetation_data"),
    ("ballast",                 "ballast_data"),
    ("vegetation",              "vegetation_data"),
    ("sod",                     "sod_data"),
    ("geometry",                "geometry_data"),
    ("gauge",                   "gauge_data"),
    ("squat",                   "squat_data"),
    ("corrugation",             "corrugation_data"),
    ("head check",              "head_check_data"),
]

In [4]:
_SKIP_ROW_PATTERNS = re.compile(
    r"reporting\s*date|itms\s*reports?|^reporting|^iTMS",
    re.IGNORECASE,
)

In [5]:
def _is_numeric(val) -> bool:
    """Return True if val can be cast to float."""
    try:
        float(str(val).strip())
        return True
    except (ValueError, TypeError):
        return False
 
 
def _cell_lines(val) -> list:
    """Split a cell value on newlines, return non-empty stripped lines."""
    if pd.isna(val):
        return []
    return [ln.strip() for ln in str(val).split("\n") if ln.strip()]
 
 
def _populated(row: pd.Series) -> list:
    """Non-empty string values from a row."""
    return [
        str(v).strip()
        for v in row.values
        if pd.notna(v) and str(v).strip() not in ("", "nan")
    ]

In [6]:
def _is_skip_row(pop: list) -> bool:
    """
    Return True if this row should be ignored while building the header.
    Covers:
      • Rows where every cell matches a skip-pattern (Reporting Date, iTMS, …)
      • Single-cell rows with very long text (metadata / EXCEPTION REPORT cell)
    """
    if not pop:
        return True
    if all(_SKIP_ROW_PATTERNS.search(p) for p in pop):
        return True
    if len(pop) == 1 and len(pop[0]) > 50:
        return True
    return False

In [7]:
def detect_blocks(df: pd.DataFrame) -> list:
    """
    Find every row containing EXCEPTION REPORT.
    Slice the sheet into one sub-DataFrame per block.
    Everything above the first trigger row is ignored.
    """
    trigger_rows = []
    for idx, row in df.iterrows():
        row_text = " ".join(str(v) for v in row.values if pd.notna(v))
        if BLOCK_TRIGGER.lower() in row_text.lower():
            trigger_rows.append(idx)
 
    if not trigger_rows:
        return []
 
    blocks = []
    for i, start in enumerate(trigger_rows):
        end = trigger_rows[i + 1] if i + 1 < len(trigger_rows) else df.index[-1] + 1
        block = df.loc[start : end - 1].reset_index(drop=True)
        blocks.append(block)
 
    log.info(f"  Detected {len(blocks)} block(s).")
    return blocks

In [8]:
def extract_metadata(block: pd.DataFrame) -> dict:
    """
    FIX-1: Split every cell on \\n before joining into full_text.
    This exposes metadata embedded inside multi-line cells.
    All 8 file types store section, TRC No, Run Date etc. inside a
    single multi-line cell — splitting on \\n makes them regex-accessible.
    """
    lines = []
    for _, row in block.iloc[:HEADER_SCAN_ROWS].iterrows():
        for val in row.values:
            lines.extend(_cell_lines(val))
 
    full_text = " | ".join(lines)
 
    def find(pattern, default=""):
        m = re.search(pattern, full_text, re.IGNORECASE)
        return m.group(1).strip() if m else default
 
    # ── Full section string e.g. "Kalyan-Manmad Line: UP Line KM : 139 to 261"
    section_full = find(r"section[:\s]*([^\|]+?)(?:\s*\||$)")
 
    # section_name: everything up to the direction keyword
    section_name = ""
    m_name = re.search(
        r"^(.*?Line)\s*:\s*(?:UP|DN|Down|IInd|IIIrd|IVth|3rd|4th)\s",
        section_full, re.IGNORECASE
    )
    if m_name:
        section_name = m_name.group(1).strip()
    else:
        m_fb = re.search(r"^(.*?)(?:\s*KM\s*:|$)", section_full, re.IGNORECASE)
        section_name = m_fb.group(1).strip() if m_fb else section_full
 
    # line_direction: UP / DN / Down / IIIrd / IVth etc.
    line_direction = ""
    m_dir = re.search(
        r"\b(UP|DN|Down|IInd|IIIrd|IVth|3rd|4th)\s+Line",
        section_full, re.IGNORECASE
    )
    if m_dir:
        line_direction = m_dir.group(1).strip()
 
    km_range  = find(r"KM\s*:\s*([\d.]+\s*to\s*[\d.]+)")
    trc_no    = find(r"TRC\s*No\.?\s*:?\s*([A-Z0-9\-/]+)")
    run_date  = find(r"RUN\s*Date\s*:?\s*([0-9]{1,2}[-/\s]\w+[-/\s][0-9]{2,4})")
    run_no    = find(r"RUN\s*No\.?\s*:?\s*([^\s\|]+)")
    rail_side = find(r"\b(Left Rail|Right Rail|Left|Right|LH|RH)\b")
    defect    = find(r"DEFECTS[-\s]+([A-Z][A-Z\s&/]+?)(?:\s*\(|\s*\||$)")
    if not defect:
        defect = find(r"[Dd]efect\s*[=:]\s*([^\|\n]+")
 
    return {
        "full_header_text": full_text,
        "section_name":     section_name,
        "line_direction":   line_direction,
        "km_range":         km_range,
        "trc_no":           trc_no,
        "run_date":         run_date,
        "run_no":           run_no,
        "defect":           defect,
        "rail_side":        rail_side,
    }

In [9]:
def _build_multilevel_columns(header_rows: list, n_cols: int) -> list:
    """
    FIX-2: Combine multiple header rows into descriptive column names.
 
    Rules:
    • All rows EXCEPT the last are forward-filled (handles merged cells in
      grouping / section rows like "Location", "Component Defects" etc.).
    • The LAST row is NOT forward-filled — leaf-level sub-headers like
      "Km", "Meter" should not bleed into separator NaN columns.
    • Unique non-empty parts are joined with '_' per column position.
 
    Examples:
      Fittings  → location_km | location_block |
                  component_defects_left_rail_missing_loose_clip | …
      Sleeper   → location_km | location_block |
                  nos_of_affected_sleepers_broken_sleeper | …
      Rail Def  → s_no | location_km | location_meter |
                  value_of_defect_mm | … (NaN separator col dropped later)
    """
    # Pad all rows to n_cols
    padded = []
    for row_vals in header_rows:
        row = list(row_vals)
        row = row[:n_cols] + [None] * max(0, n_cols - len(row))
        padded.append(row)
 
    # Forward-fill all rows EXCEPT the last one
    ff_rows = []
    for idx, row in enumerate(padded):
        sr = pd.Series(row, dtype=object).replace({"": None, "nan": None})
        if idx < len(padded) - 1:   # not the last header row → forward-fill
            sr = sr.ffill()
        ff_rows.append(sr.tolist())
 
    # Build column name by collecting unique parts top-to-bottom
    result = []
    for col_idx in range(n_cols):
        parts, prev = [], None
        for row in ff_rows:
            val = row[col_idx] if col_idx < len(row) else None
            if val and str(val).strip() and str(val).strip().lower() != "nan":
                v = str(val).strip()
                if v != prev:
                    parts.append(v)
                    prev = v
        result.append("_".join(parts) if parts else f"col_{col_idx}")
 
    return result

In [10]:
def _find_header_and_data(block: pd.DataFrame):
    """
    FIX-2: Scan the block (starting after the EXCEPTION REPORT row) to
    separate metadata rows, table header rows, and data rows.
 
    Skips:
      • Single-cell long-text rows (metadata, EXCEPTION REPORT line itself)
      • Rows where every populated cell matches a skip pattern
        (Reporting Date, iTMS Reports — even when they span multiple columns)
      • Blank rows
 
    Collects rows as header rows until the first row whose FIRST non-empty
    cell is numeric → everything from that row onwards is data.
 
    Returns (list_of_header_row_value_lists, data_start_iloc_index).
    """
    header_rows = []
    data_start  = None
 
    for i in range(1, len(block)):          # row 0 is the trigger
        row = block.iloc[i]
        pop = _populated(row)
 
        if not pop:
            continue                        # blank row — skip
 
        if _is_skip_row(pop):
            continue                        # metadata / reporting-date — skip
 
        # First non-empty cell is numeric → data starts here
        if _is_numeric(pop[0]):
            data_start = i
            break
 
        header_rows.append(row.tolist())    # genuine header row
 
    return header_rows, data_start

In [11]:
def extract_table(block: pd.DataFrame):
    """
    Locate header rows and data rows; build a DataFrame with descriptive
    column names.  Returns None if no valid table is found.
    """
    header_rows, data_start = _find_header_and_data(block)
 
    if not header_rows or data_start is None:
        log.warning("  Could not locate table header or data — skipping block.")
        return None
 
    n_cols    = len(block.columns)
    col_names = _build_multilevel_columns(header_rows, n_cols)
 
    data_df = block.iloc[data_start:].reset_index(drop=True)
 
    # Align col_names width to DataFrame
    while len(col_names) < len(data_df.columns):
        col_names.append(f"col_{len(col_names)}")
    data_df.columns = col_names[: len(data_df.columns)]
 
    # FIX-5: Drop footer rows where the lead cell is not a real number.
    # Use .iloc[:, 0] (not column name) to avoid crash when duplicate col names exist
    # before deduplication (e.g. Rail Defects has two "S.No." columns).
    mask = data_df.iloc[:, 0].apply(_is_numeric)
    data_df = data_df[mask.values].reset_index(drop=True)
 
    return data_df if not data_df.empty else None

In [12]:
def clean_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    • Drop all-NaN rows and all-NaN columns.
    • Remove auto-generated / unnamed columns.
    • Standardise column names to snake_case.
    • FIX-3: Deduplicate column names.
    • FIX-5 second pass: keep only rows where lead column is numeric.
    """
    df = df.dropna(how="all").reset_index(drop=True)
    df = df.dropna(how="all", axis=1)           # drop fully-empty columns
 
    # Remove auto-generated columns
    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed|^col_\d+$", na=False)]
 
    # snake_case
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^\w]", "_", regex=True)
        .str.replace(r"_+",   "_", regex=True)
        .str.strip("_")
    )
    df = df.loc[:, df.columns != ""]
 
    # FIX-3: Deduplicate column names
    seen, new_cols = {}, []
    for col in df.columns:
        if col in seen:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            new_cols.append(col)
    df.columns = new_cols
 
    # FIX-5 second pass: enforce numeric lead column
    if len(df.columns) > 0:
        mask = df.iloc[:, 0].apply(_is_numeric)
        df = df[mask.values].reset_index(drop=True)
 
    return df

In [13]:
def detect_dataset_type(metadata: dict, file_name: str = "") -> str:
    """
    FIX-6: Filename checked first, then header / defect text.
    Returns the matching table name or 'general_data'.
    """
    text = (
        file_name + " " +
        metadata.get("full_header_text", "") + " " +
        metadata.get("defect", "")
    ).lower()
 
    for keyword, table_name in DATASET_TYPE_MAP:
        if keyword in text:
            return table_name
 
    return "general_data"

In [14]:
def _table_columns(conn: sqlite3.Connection, table_name: str) -> list:
    cur = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name=?",
        (table_name,),
    )
    if cur.fetchone() is None:
        return []
    return [r[1] for r in conn.execute(f"PRAGMA table_info('{table_name}')")]
 
 
def _add_missing_columns(conn: sqlite3.Connection, table_name: str, cols: list):
    existing = _table_columns(conn, table_name)
    for col in cols:
        if col not in existing:
            try:
                conn.execute(f"ALTER TABLE '{table_name}' ADD COLUMN '{col}' TEXT")
            except Exception as e:
                log.warning(f"  Could not add column '{col}': {e}")

In [15]:
def store_to_sql(df: pd.DataFrame, table_name: str, db_path: str = DB_PATH):
    """
    FIX-4: Append-if-exists / create-if-not.
 
    First run  → table does not exist  → pandas CREATE TABLE + INSERT.
    Later runs → table already exists  → ALTER TABLE for new cols + INSERT.
 
    Data is NEVER overwritten — only appended.
    """
    try:
        with sqlite3.connect(db_path) as conn:
            existing = _table_columns(conn, table_name)
 
            if not existing:
                df.to_sql(table_name, conn, if_exists="append", index=False)
            else:
                _add_missing_columns(conn, table_name, df.columns.tolist())
                existing = _table_columns(conn, table_name)
                common = [c for c in df.columns if c in existing]
                if not common:
                    log.warning(f"  No common columns for '{table_name}' — skipping.")
                    return
                df[common].to_sql(table_name, conn, if_exists="append", index=False)
 
        log.info(f"  Stored {len(df)} row(s) → '{table_name}'")
 
    except Exception as e:
        log.error(f"  DB write error for '{table_name}': {e}")


In [16]:
def process_folder(folder_path: str, db_path: str = DB_PATH):
    """
    Walk every .xlsx in folder_path.
 
    For each file → sheet → EXCEPTION REPORT block:
      1. extract_metadata   — section, TRC No, Run Date, Run No, defect, rail side
      2. extract_table      — universal multi-level header + data extraction
      3. clean_table        — snake_case columns, dedup, numeric filter, drop empty cols
      4. attach metadata    — section etc. added as columns on every row
      5. detect_dataset_type — filename-first lookup → table name
      6. store_to_sql       — append-if-exists / create-if-not
    """
    folder = Path(folder_path)
    xlsx_files = sorted(folder.glob("*.xlsx"))
 
    if not xlsx_files:
        log.warning(f"No .xlsx files found in: {folder_path}")
        return
 
    log.info(f"Found {len(xlsx_files)} Excel file(s) in '{folder_path}'.")
 
    total_blocks = 0
    total_rows   = 0
    summary: dict = {}
 
    for file_path in xlsx_files:
        log.info(f"\n{'='*60}")
        log.info(f"FILE: {file_path.name}")
 
        try:
            xl = pd.ExcelFile(file_path, engine="openpyxl")
        except Exception as e:
            log.error(f"  Cannot open '{file_path.name}': {e}")
            continue
 
        for sheet_name in xl.sheet_names:
            log.info(f"  Sheet: '{sheet_name}'")
            try:
                raw_df = xl.parse(sheet_name, header=None, dtype=str)
            except Exception as e:
                log.error(f"  Cannot read '{sheet_name}': {e}")
                continue
 
            if raw_df.empty:
                continue
 
            blocks = detect_blocks(raw_df)
            if not blocks:
                continue
 
            for b_idx, block in enumerate(blocks):
                log.info(f"  Block {b_idx + 1}/{len(blocks)} ...")
                total_blocks += 1
                try:
                    # ── 1. Metadata ───────────────────────────────────
                    metadata = extract_metadata(block)
                    log.info(
                        f"    section_name='{metadata['section_name']}'  "
                        f"trc='{metadata['trc_no']}'  "
                        f"date='{metadata['run_date']}'"
                    )
 
                    # ── 2. Table extraction ───────────────────────────
                    table_df = extract_table(block)
                    if table_df is None or table_df.empty:
                        log.warning(f"    No usable table — skipping.")
                        continue
 
                    # ── 3. Clean ──────────────────────────────────────
                    table_df = clean_table(table_df)
                    if table_df.empty:
                        log.warning(f"    Empty after cleaning — skipping.")
                        continue
 
                    # ── 4. Attach metadata columns ────────────────────
                    table_df["section_name"]   = metadata.get("section_name",   "")
                    table_df["line_direction"] = metadata.get("line_direction", "")
                    table_df["km_range"]       = metadata.get("km_range",       "")
                    table_df["trc_no"]         = metadata.get("trc_no",         "")
                    table_df["run_date"]       = metadata.get("run_date",       "")
                    table_df["run_no"]         = metadata.get("run_no",         "")
                    table_df["defect"]         = metadata.get("defect",         "")
                    table_df["rail_side"]      = metadata.get("rail_side",      "")
                    table_df["source_file"] = file_path.name
                    table_df["sheet_name"]  = sheet_name
 
                    # ── 5. Detect table name ──────────────────────────
                    tbl = detect_dataset_type(metadata, file_path.name)
                    log.info(f"    → table: '{tbl}'  rows: {len(table_df)}")
 
                    # ── 6. Write to SQLite ────────────────────────────
                    store_to_sql(table_df, tbl, db_path)
 
                    total_rows += len(table_df)
                    summary[tbl] = summary.get(tbl, 0) + len(table_df)
 
                except Exception as e:
                    log.error(f"    Block {b_idx + 1} failed: {e}", exc_info=True)
 
    # ── Final summary ──────────────────────────────────────────────────
    log.info(f"\n{'='*60}")
    log.info("PIPELINE COMPLETE")
    log.info(f"  Total blocks   : {total_blocks}")
    log.info(f"  Total rows     : {total_rows}")
    log.info(f"  Tables written :")
    for tbl, rows in sorted(summary.items()):
        log.info(f"    {tbl:<42} {rows:>6} rows")
    log.info(f"  Database       : {db_path}")
    log.info("=" * 60)

In [17]:
# AFTER (hardcoded path)
if __name__ == "__main__":

    input_folder = r"D:\\ENGINEER\\IndianRailwaysProject\\data"   # ← paste your folder path here
    output_db    = "railway.db"                         # ← DB will be created here

    process_folder(input_folder, output_db)

2026-04-10 00:00:08,143 [INFO] Found 8 Excel file(s) in 'D:\\ENGINEER\\IndianRailwaysProject\\data'.
2026-04-10 00:00:08,144 [INFO] 
2026-04-10 00:00:08,144 [INFO] FILE: 1 SOD exception.xlsx
2026-04-10 00:00:08,539 [INFO]   Sheet: 'KYN-MMR UP'
2026-04-10 00:00:08,542 [INFO]   Sheet: 'KYN-MMR DN'
2026-04-10 00:00:08,544 [INFO]   Detected 1 block(s).
2026-04-10 00:00:08,544 [INFO]   Block 1/1 ...
2026-04-10 00:00:08,546 [ERROR]     Block 1 failed: missing ), unterminated subpattern at position 19
Traceback (most recent call last):
  File "C:\Users\Ajitesh Channa\AppData\Local\Temp\ipykernel_33772\1276306770.py", line 56, in process_folder
    metadata = extract_metadata(block)
               ^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Ajitesh Channa\AppData\Local\Temp\ipykernel_33772\2759429423.py", line 50, in extract_metadata
    defect = find(r"[Dd]efect\s*[=:]\s*([^\|\n]+")
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Ajitesh Channa\AppData\Local\Temp\ipykernel_33